# Grover's search in OpenQASM 3

Mark |101> among 8 states using only vocabulary gates; validate against the ideal closed-form amplitudes.


In [ ]:
import math
from qvm.qasm3_parser import OpenQASM3Parser
from qvm.simulator import Simulator
import numpy as np

ORACLE = "x q[1]; h q[1]; ccx q[0], q[2], q[1]; h q[1]; x q[1];"
DIFFUSER = ("h q[0]; h q[1]; h q[2]; x q[0]; x q[1]; x q[2];"
            "h q[2]; ccx q[0], q[1], q[2]; h q[2];"
            "x q[0]; x q[1]; x q[2]; h q[0]; h q[1]; h q[2];")
N_ITER = int(math.floor(math.pi / 4 * math.sqrt(8)))

body = "h q[0]; h q[1]; h q[2];\n" + (ORACLE + DIFFUSER) * N_ITER
src = 'OPENQASM 3.0;\ninclude "stdgates.inc";\nqubit[3] q;\n' + body
grover = OpenQASM3Parser().parse(src)

state, _ = Simulator().simulate(grover)
probs = np.abs(state) ** 2
print({format(i, "03b"): round(p, 3) for i, p in enumerate(probs) if p > 0.02})

In [ ]:
# closed-form reference: two Grover iterations from uniform
amps = np.full(8, 1 / math.sqrt(8))
for _ in range(N_ITER):
    amps[5] *= -1                      # mark |101>
    amps = 2 * amps.mean() - amps      # invert about mean
assert np.allclose(probs, np.abs(amps) ** 2, atol=1e-9)
print("matches ideal Grover amplification")